# Iceberg Basics — 03: Snapshots

## What you will learn
Snapshots are the foundation of Iceberg's time travel, ACID guarantees,
and incremental processing capabilities.

In this notebook:
1. What a snapshot is — atomic point-in-time view
2. How snapshots are created — one per write operation
3. Querying the `.snapshots` metadata table
4. Snapshot operations — `rollback`, `set-current`, `cherry-pick`
5. Snapshot expiry — managing storage growth
6. Incremental reads between snapshots


In [ ]:
import os, time, pathlib, shutil, random, datetime
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

MASTER   = 'spark://spark-master:7077'
DATA_DIR = '/workspace/data/iceberg_basics'
pathlib.Path(DATA_DIR).mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName('iceberg-basics')
    .master(MASTER)
    .config('spark.executor.memory', '2g')
    .config('spark.driver.memory',   '1g')
    .config('spark.executor.cores',  '2')
    .config('spark.shuffle.sort.bypassMergeThreshold', '0')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
spark.sql("CREATE DATABASE IF NOT EXISTS local.icedb")
print(f"Spark {spark.version} | Iceberg catalog ready")

random.seed(42)
N = 100_000
CATEGORIES = ["Electronics","Clothing","Books","Food","Sports","Furniture"]
COUNTRIES  = ["US","UK","DE","FR","JP","CA","AU","BR"]
STATUSES   = ["pending","confirmed","shipped","delivered","cancelled"]

schema = StructType([
    StructField("order_id",    LongType()),
    StructField("customer_id", LongType()),
    StructField("product",     StringType()),
    StructField("category",    StringType()),
    StructField("country",     StringType()),
    StructField("quantity",    IntegerType()),
    StructField("price",       DoubleType()),
    StructField("revenue",     DoubleType()),
    StructField("status",      StringType()),
    StructField("order_date",  DateType()),
])

base = datetime.date(2024, 1, 1)
rows = []
for i in range(N):
    qty   = random.randint(1, 10)
    price = round(random.uniform(5, 1000), 2)
    d     = base + datetime.timedelta(days=random.randint(0, 364))
    rows.append((i+1, random.randint(1,10000),
                 f"Product_{random.randint(1,200)}",
                 random.choice(CATEGORIES), random.choice(COUNTRIES),
                 qty, price, round(qty*price, 2),
                 random.choice(STATUSES), d))

df = spark.createDataFrame(rows, schema)
df.cache().count()
print(f"Dataset ready: {N:,} rows")

In [ ]:
spark.sql("DROP TABLE IF EXISTS local.icedb.snap_orders")
df.writeTo("local.icedb.snap_orders").using("iceberg").createOrReplace()
print(f"Table created: {spark.table('local.icedb.snap_orders').count():,} rows")

## Step 1 — Creating and Inspecting Snapshots


In [ ]:
# Each write = new snapshot
df.limit(1000).writeTo("local.icedb.snap_orders").append()
print("Appended 1000 rows → new snapshot")

spark.sql("UPDATE local.icedb.snap_orders SET status='shipped' WHERE status='confirmed' AND order_id % 5 = 0")
print("Updated rows → new snapshot")

spark.sql("DELETE FROM local.icedb.snap_orders WHERE status='cancelled' AND price < 30")
print("Deleted rows → new snapshot")

print()
print("All snapshots:")
spark.sql("""
    SELECT snapshot_id,
           committed_at,
           operation,
           summary['added-records']   AS added,
           summary['deleted-records'] AS deleted,
           summary['total-records']   AS total
    FROM local.icedb.snap_orders.snapshots
    ORDER BY committed_at
""").show(truncate=False)

## Step 2 — Time Travel via Snapshots


In [ ]:
snaps = spark.sql("""
    SELECT snapshot_id, committed_at, operation,
           summary['total-records'] AS total
    FROM local.icedb.snap_orders.snapshots
    ORDER BY committed_at
""").collect()

print("Row count at each snapshot:")
for row in snaps:
    sid   = row["snapshot_id"]
    total = spark.read.option("snapshot-id", sid).table("local.icedb.snap_orders").count()
    print(f"  snap {sid}  {row['operation']:<15} {total:,} rows")

In [ ]:
# Rollback to first snapshot
first_snap = snaps[0]["snapshot_id"]
print(f"Rolling back to first snapshot: {first_snap}")

spark.sql(f"""
    CALL local.system.rollback_to_snapshot(
        table => 'local.icedb.snap_orders',
        snapshot_id => {first_snap}
    )
""")

current_count = spark.table("local.icedb.snap_orders").count()
print(f"After rollback: {current_count:,} rows  (back to initial state)")
print()
print("Rollback creates a NEW snapshot pointing to old data files.")
print("All history is preserved — you can rollback the rollback!")
spark.sql("SELECT snapshot_id, operation FROM local.icedb.snap_orders.snapshots ORDER BY committed_at").show()

## Step 3 — Snapshot Expiry


In [ ]:
# Expire old snapshots to free metadata space
# (does NOT delete data files — use remove_orphan_files for that)
print("Before expiry:")
snap_count_before = spark.sql("SELECT COUNT(*) FROM local.icedb.snap_orders.snapshots").collect()[0][0]
print(f"  Snapshot count: {snap_count_before}")

spark.sql("""
    CALL local.system.expire_snapshots(
        table         => 'local.icedb.snap_orders',
        retain_last   => 2
    )
""").show()

snap_count_after = spark.sql("SELECT COUNT(*) FROM local.icedb.snap_orders.snapshots").collect()[0][0]
print(f"After expiry (retain_last=2): {snap_count_after} snapshots remain")
print()
print("Expired snapshot metadata is removed.")
print("Data files still exist until remove_orphan_files is called.")

## Summary

```python
# Inspect snapshots
spark.sql("SELECT * FROM local.db.table.snapshots")

# Read at specific snapshot
spark.read.option("snapshot-id", snap_id).table("local.db.table")

# Rollback to snapshot
spark.sql(f"CALL local.system.rollback_to_snapshot(table => 'db.table', snapshot_id => {snap_id})")

# Expire old snapshots
spark.sql("CALL local.system.expire_snapshots(table => 'db.table', retain_last => 5)")
```

Every write operation creates a new snapshot. Snapshots enable:
- **Time travel** — query any past state
- **Rollback** — undo bad writes
- **Incremental reads** — process only new data since last snapshot
- **Audit trail** — see history of all operations
